# Investimento Simples

Mude so estes tres valores e rode as celulas.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

capital_inicial = 1000.0  # dinheiro no comeco
aporte_mensal = 500.0  # valor colocado por mes
tempo_total = 5.0  # tempo da simulacao em anos

taxas = [0.0925, 0.1375, 0.1175, 0.1225, 0.1500, 0.1500]  # taxas anuais
anos_taxas = ["2021", "2022", "2023", "2024", "2025", "2026"]  # nomes dos anos


In [ ]:
def reais(x):
    texto = f"{x:,.2f}"  # formata o numero
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")  # troca para o padrao brasileiro


meses = max(1, int(round(tempo_total * 12)))  # transforma anos em meses
tempo = np.arange(meses + 1) / 12  # cria o eixo do tempo
saldo = np.zeros(meses + 1)  # guarda o saldo com rendimento
investido = np.zeros(meses + 1)  # guarda o total aportado

saldo[0] = capital_inicial  # valor inicial do saldo
investido[0] = capital_inicial  # valor inicial investido

for mes in range(meses):
    indice_ano = min(mes // 12, len(taxas) - 1)  # escolhe a taxa do ano
    taxa_anual = taxas[indice_ano]  # pega a taxa anual
    taxa_mensal = (1 + taxa_anual) ** (1 / 12) - 1  # converte para taxa mensal

    saldo[mes + 1] = saldo[mes] * (1 + taxa_mensal) + aporte_mensal  # rende e soma aporte
    investido[mes + 1] = investido[mes] + aporte_mensal  # soma o aporte


In [ ]:
valor_final = saldo[-1]  # ultimo saldo calculado
total_investido = investido[-1]  # total colocado pelo usuario
rendimento = valor_final - total_investido  # diferenca entre saldo e aportes

print("Resumo\n")
print(f"Capital inicial: R$ {reais(capital_inicial)}")
print(f"Aporte mensal:   R$ {reais(aporte_mensal)}")
print(f"Tempo total:     {tempo_total:.2f} anos")
print(f"Total investido: R$ {reais(total_investido)}")
print(f"Valor final:     R$ {reais(valor_final)}")
print(f"Rendimento:      R$ {reais(rendimento)}")


In [ ]:
print("Taxas anuais usadas")
for i in range(max(1, int(np.ceil(tempo_total)))):
    indice_ano = min(i, len(taxas) - 1)  # repete a ultima taxa se faltar ano
    print(f"{anos_taxas[indice_ano]}: {100 * taxas[indice_ano]:.2f}% ao ano")


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(tempo, investido, lw=2.4, label="Total investido")  # linha do dinheiro aplicado
plt.plot(tempo, saldo, lw=2.4, label="Saldo estimado")  # linha do saldo final
plt.fill_between(tempo, investido, saldo, where=saldo >= investido, alpha=0.2, label="Rendimento")  # area de ganho
plt.title("Simulacao simples de investimento")
plt.xlabel("Tempo (anos)")
plt.ylabel("Valor (R$)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Parte com alpha

Aqui entramos no modelo generalizado

`dP/dt = r * P^alpha`

para comparar tres regimes:

- `alpha < 1`: crescimento subexponencial;
- `alpha = 1`: crescimento exponencial;
- `alpha > 1`: crescimento superacelerado.


In [ ]:
taxa_alpha = 0.12  # taxa usada no modelo com alpha
alphas = [0.5, 1.0, 1.2]  # valores para comparar regimes
tempo_alpha = np.linspace(0, tempo_total, 400)  # eixo do tempo para os graficos


In [ ]:
def modelo_alpha(P0, r, alpha, t):
    if np.isclose(alpha, 1.0):
        return P0 * np.exp(r * t)  # caso exponencial

    base = P0 ** (1 - alpha) + (1 - alpha) * r * t  # parte interna da formula
    curva = np.full_like(t, np.nan, dtype=float)  # comeca com valores invalidos
    ok = base > 0  # a formula so vale quando a base eh positiva
    curva[ok] = base[ok] ** (1 / (1 - alpha))  # aplica a potencia apenas onde pode
    return curva


def limite_alpha(P0, r, alpha):
    if np.isclose(alpha, 1.0) or np.isclose(r, 0.0):
        return np.inf  # nao ha corte no dominio nesses casos

    base0 = P0 ** (1 - alpha)  # valor inicial da base
    coef = (1 - alpha) * r  # coeficiente do tempo

    if coef < 0:
        return base0 / (-coef)  # tempo limite quando a base zera

    return np.inf


In [ ]:
print("Resumo do modelo com alpha")
for alpha in alphas:
    limite = limite_alpha(capital_inicial, taxa_alpha, alpha)  # calcula o limite do dominio
    valor_final = modelo_alpha(capital_inicial, taxa_alpha, alpha, tempo_alpha)[-1]  # pega o ultimo valor

    if np.isfinite(limite):
        dominio = f"0 <= t < {limite:.2f}"  # dominio finito
    else:
        dominio = "t >= 0"  # dominio aberto para frente

    if np.isnan(valor_final):
        texto_final = "fora do dominio"  # avisa se o tempo escolhido passou do limite
    else:
        texto_final = f"R$ {reais(valor_final)}"  # mostra o valor final

    print(f"alpha = {alpha:.1f} | dominio: {dominio} | valor final: {texto_final}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for alpha in alphas:
    curva = modelo_alpha(capital_inicial, taxa_alpha, alpha, tempo_alpha)  # curva no tempo
    ax1.plot(tempo_alpha, curva, lw=2.4, label=f"alpha = {alpha}")  # plota P(t)

    limite = limite_alpha(capital_inicial, taxa_alpha, alpha)  # pega o limite do dominio
    if np.isfinite(limite) and 0 < limite < tempo_total:
        ax1.axvline(limite, color="gray", ls="--", alpha=0.5)  # marca o corte do dominio

ax1.set_title("Curvas P(t) para diferentes alpha")
ax1.set_xlabel("Tempo (anos)")
ax1.set_ylabel("Capital")
ax1.grid(alpha=0.3)
ax1.legend()

P = np.linspace(0, 2 * capital_inicial, 400)  # grade de capital para o diagrama de fase
for alpha in alphas:
    dPdt = taxa_alpha * P ** alpha  # taxa de crescimento em funcao do capital
    ax2.plot(P, dPdt, lw=2.4, label=f"alpha = {alpha}")  # plota dP/dt por P

ax2.set_title("Diagrama de fase: dP/dt por P")
ax2.set_xlabel("P")
ax2.set_ylabel("dP/dt")
ax2.grid(alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()
